In [ ]:
#!pip install --upgrade transformers

In [21]:
from transformers import pipeline
import json
from pathlib import Path

import pandas as pd

In [22]:
import transformers
print(transformers.__version__)

5.4.0


In [23]:
# ── 1. Load model ─────────────────────────────────────────────────────────────
# Good small options to try:
# - "mistralai/Mistral-7B-Instruct-v0.3"
# - "Qwen/Qwen2.5-7B-Instruct"
# - "BioMistral/BioMistral-7B" (medical-focused)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

pipe = pipeline(
    "text-generation",
    model=MODEL_NAME,
    device_map="auto",      # automatically uses GPU if available, else CPU
    max_new_tokens=1024,
)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [29]:

# ── 2. Prompt ─────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a medical information extraction assistant.
Given a medical text, extract structured information and return it as JSON only,
with no preamble or explanation.
"""

def build_prompt(text: str) -> str:
    return f"""Extract the following fields from the medical text below.
If a field is not mentioned, use null.

Fields to extract:
- circumstances: the cause of the incident which lead to the ER visit
- symptoms: list of symptoms reported by the patient
- diagnosis: the diagnosis if mentioned
- medications: list of medications mentioned
- patient_age: age of the patient if mentioned
- patient_gender: gender of the patient if mentioned

Return only a JSON object with these fields.

Medical text:
{text}
"""

def build_prompt_normalized(text: str) -> str:
    return f"""Extract clinical information and normalize it using standard medical terminology.

- Map symptoms and diagnoses to standardized terms (e.g., SNOMED-like labels if possible).
- Normalize medications to their generic names.
- Normalize age to integer if possible.

Return JSON with:
- symptoms_normalized
- diagnosis_normalized
- medications_normalized

Medical text:
{text}
"""


def build_prompt_timeline(text: str) -> str:
    return f"""Extract a timeline of events from the medical text.

Return a JSON list of events in chronological order:
- event
- time_expression (exact or relative, e.g., "2 days ago")

Medical text:
{text}
"""

def build_prompt_causality(text: str) -> str:
    return f"""Identify causal relationships in the medical text.

Return JSON:
- cause: what triggered the condition or ER visit
- effect: resulting symptoms or diagnosis
- confidence: high / medium / low

Medical text:
{text}
"""

def build_prompt_negation(text: str) -> str:
    return f"""Extract symptoms and indicate whether they are:
- present
- absent (negated)
- uncertain

Return JSON list:
- symptom
- status (present / absent / uncertain)

Medical text:
{text}
"""

def build_prompt_clinical_summary(text: str) -> str:
    return f"""Summarize the medical text for a physician.

- Use precise clinical language
- Be concise
- Include key findings only

Medical text:
{text}
"""


def build_prompt_patient_summary(text: str) -> str:
    return f"""Explain the medical text to a patient with no medical background.

- Use simple language
- Avoid jargon
- Be reassuring but accurate

Medical text:
{text}
"""

def build_prompt_triage(text: str) -> str:
    return f"""Classify the urgency level of this case.

Return JSON:
- triage_level: (low / moderate / high / emergency)
- justification: short explanation

Medical text:
{text}
"""


def build_prompt_inconsistency(text: str) -> str:
    return f"""Detect inconsistencies or contradictions in the medical text.

Return JSON list:
- statement_1
- statement_2
- explanation

Medical text:
{text}
"""


def build_prompt_missing(text: str) -> str:
    return f"""Identify missing but clinically relevant information.

Return JSON list of missing fields (e.g., allergies, medications, history).

Medical text:
{text}
"""


def build_prompt_relations(text: str) -> str:
    return f"""Extract relationships between entities.

Return JSON list:
- subject
- relation (e.g., "has_symptom", "treated_with")
- object

Medical text:
{text}
"""



def build_prompt_noisy(text: str) -> str:
    return f"""Extract key medical information despite possible noise, typos, or incomplete sentences.

Return JSON:
- symptoms
- diagnosis
- medications

Medical text:
{text}
"""


def build_prompt_counterfactual(text: str) -> str:
    return f"""Based on the medical text, answer:

"What would likely change if the patient had NOT received the mentioned treatment?"

Provide a short explanation.

Medical text:
{text}
"""


def build_prompt_sensitivity(text: str) -> str:
    return f"""Extract diagnosis and symptoms.

Be precise: small wording differences matter.

Return JSON:
- symptoms
- diagnosis

Medical text:
{text}
"""

In [30]:

# ── 3. Extraction function ────────────────────────────────────────────────────
def extract_medical_info(text: str) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_prompt(text)}
    ]
    output = pipe(messages)
    raw = output[0]["generated_text"][-1]["content"].strip()

    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        # Return raw text if JSON parsing fails, so you can inspect it
        print(f"Warning: could not parse JSON, returning raw output.")
        return {"raw_output": raw}



In [31]:
# ── 4. Process a folder of .txt files ────────────────────────────────────────
# Test on just the first 5 files
def process_er_folder(folder_path: str, limit: int = None) -> pd.DataFrame:
    records = []
    txt_files = list(Path(folder_path).glob("*.txt"))
    if limit:
        txt_files = txt_files[:limit]
    print(f"Processing {len(txt_files)} files...")

    for i, txt_file in enumerate(txt_files):
        print(f"[{i+1}/{len(txt_files)}] {txt_file.name}...")
        text = txt_file.read_text(encoding="utf-8").strip()
        if not text:
            continue
        extracted = extract_medical_info(text)
        extracted["source_file"] = txt_file.name
        records.append(extracted)

    return pd.DataFrame(records)


In [32]:
# Try 5 first
df_test = process_er_folder("mtsamples_er/texts", limit=5)
print(df_test)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing 5 files...
[1/5] Sample_Name Abdominal_Pain_-_Consult
Medical_Specialty 
			__
				Emergency_Room_Reports.txt...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2/5] Sample_Name Abrasions_&_Lacerations_-_ER_Visit
Medical_Specialty 
			__
				Emergency_Room_Reports.txt...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[3/5] Sample_Name Accidental_Celesta_Ingestion_-_ER_Visit
Medical_Specialty 
			__
				Emergency_Room_Reports.txt...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[4/5] Sample_Name Acute_Inferior_Myocardial_Infarction
Medical_Specialty 
			__
				Emergency_Room_Reports.txt...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[5/5] Sample_Name Agitation_-_ER_Visit
Medical_Specialty 
			__
				Emergency_Room_Reports.txt...
                                       circumstances  \
0            Abdominal pain, persistent for 7-8 days   
1  A 12-year-old fell off his bicycle, not wearin...   
2   Accidental ingestion of Celesta 40 mg per tablet   
3                                         Chest pain   
4  The patient was brought in by family as she fe...   

                                            symptoms  \
0  [abdominal pain, anorexia, obstipation, passin...   
1                 [neck pain, loss of consciousness]   
2                                               None   
3  [Chest pain, Shortness of breath, Diaphoresis,...   
4  [Agitation, Dry heaves, Tremor, Daughter state...   

                                           diagnosis  \
0                           [sigmoid diverticulitis]   
1  [Concussion, Facial abrasion, Scalp laceration...   
2                                               None   
3   

In [34]:
# Or look at one extracted result
print(df_test.iloc[0].to_dict())
print()
print(df_test.iloc[1].to_dict())


{'circumstances': 'Abdominal pain, persistent for 7-8 days', 'symptoms': ['abdominal pain', 'anorexia', 'obstipation', 'passing flatus'], 'diagnosis': ['sigmoid diverticulitis'], 'medications': None, 'patient_age': '71', 'patient_gender': 'female', 'source_file': 'Sample_Name\xa0Abdominal_Pain_-_Consult\nMedical_Specialty\xa0\n\t\t\t__\n\t\t\t\tEmergency_Room_Reports.txt'}

{'circumstances': 'A 12-year-old fell off his bicycle, not wearing a helmet, a few hours ago.', 'symptoms': ['neck pain', 'loss of consciousness'], 'diagnosis': ['Concussion', 'Facial abrasion', 'Scalp laceration', 'Knee abrasions'], 'medications': None, 'patient_age': '12', 'patient_gender': None, 'source_file': 'Sample_Name\xa0Abrasions_&_Lacerations_-_ER_Visit\nMedical_Specialty\xa0\n\t\t\t__\n\t\t\t\tEmergency_Room_Reports.txt'}


In [8]:
#!du -sh ~/.cache/huggingface/hub/

256G	/home/tatiana.bladier/.cache/huggingface/hub/
